In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
N = 8000  # number of claims to simulate

payers = ['Medicare', 'Medicaid', 'BCBS', 'Aetna', 'Cigna', 'UnitedHealthcare']
# baseline denial rate per payer - deliberately uneven"
payer_denial_bias = {'Medicare': 0.10, 'Medicaid': 0.20, 'BCBS': 0.14,
                      'Aetna': 0.16, 'Cigna': 0.18, 'UnitedHealthcare': 0.13}

# procedure_category: (min $, max $, extra denial risk on top of payer bias)
procedures = {
    'Cardiology': (3000, 12000, 0.05),
    'Orthopedics': (2500, 15000, 0.04),
    'Oncology': (5000, 30000, 0.03),
    'General Surgery': (4000, 20000, 0.02),
    'Radiology': (500, 4000, 0.06),
    'Physical Therapy': (100, 800, 0.08),
    'Neurology': (2000, 10000, 0.03),
    'Gastroenterology': (1500, 9000, 0.04),
}

# real CARC codes mapped to plain-English reasons
carc_map = {
    'CO-50': 'Medical necessity not established',
    'CO-97': 'Service bundled into another procedure',
    'CO-16': 'Missing/incomplete information',
    'CO-197': 'Precertification/authorization missing',
    'CO-11': 'Diagnosis inconsistent with procedure',
    'PR-1': 'Deductible amount',
    'PR-2': 'Coinsurance amount',
    'CO-29': 'Time limit for filing expired',
}

rows = []
start_date = datetime(2024, 1, 1)

for i in range(N):
    payer = np.random.choice(payers)
    procedure = np.random.choice(list(procedures.keys()))
    min_amt, max_amt, proc_risk = procedures[procedure]
    billed_amount = round(np.random.uniform(min_amt, max_amt), 2)
    submission_date = start_date + timedelta(days=int(np.random.uniform(0, 365)))

    denial_prob = payer_denial_bias[payer] + proc_risk
    is_denied = np.random.random() < denial_prob

    if is_denied:
        claim_status = np.random.choice(['Denied', 'Appealed', 'Written Off'], p=[0.55, 0.30, 0.15])
        carc_code = np.random.choice(list(carc_map.keys()))
        denial_reason = carc_map[carc_code]
        resolution_days = int(np.random.uniform(10, 60))
    else:
        claim_status = 'Paid'
        carc_code = None
        denial_reason = None
        resolution_days = int(np.random.uniform(3, 20))

    rows.append({
        'claim_id': f"CLM{100000+i}",
        'patient_id': f"PAT{np.random.randint(1, 3000):05d}",
        'payer': payer,
        'procedure_category': procedure,
        'billed_amount': billed_amount,
        'claim_status': claim_status,
        'carc_code': carc_code,
        'denial_reason': denial_reason,
        'submission_date': submission_date.date(),
        'resolution_date': (submission_date + timedelta(days=resolution_days)).date(),
    })

df = pd.DataFrame(rows)
df.to_csv('claims_denial_data.csv', index=False)
print(f"Generated {len(df)} claims. Overall denial rate: {(df['claim_status'] != 'Paid').mean():.1%}")
df.head(10)

Generated 8000 claims. Overall denial rate: 20.0%


,claim_id,patient_id,payer,procedure_category,billed_amount,claim_status,carc_code,denial_reason,submission_date,resolution_date
0,CLM100000,PAT01239,Aetna,Radiology,1142.02,Paid,None,None,2024-10-11,2024-10-21
1,CLM100001,PAT00770,BCBS,Oncology,26654.40,Paid,None,None,2024-08-07,2024-08-10
2,CLM100002,PAT02905,Aetna,Physical Therapy,248.64,Denied,PR-1,Deductible amount,2024-03-07,2024-03-17
3,CLM100003,PAT00190,Medicare,Oncology,20296.32,Paid,None,None,2024-02-20,2024-02-29
4,CLM100004,PAT00647,UnitedHealthcare,Neurology,6947.09,Paid,None,None,2024-05-19,2024-05-29
5,CLM100005,PAT02825,Cigna,Cardiology,3585.46,Paid,None,None,2024-12-12,2024-12-28
6,CLM100006,PAT00776,Medicaid,Radiology,1308.13,Paid,None,None,2024-03-28,2024-04-10
7,CLM100007,PAT02614,BCBS,Physical Therapy,373.74,Paid,None,None,2024-03-07,2024-03-17
8,CLM100008,PAT01982,Medicaid,General Surgery,6957.67,Paid,None,None,2024-12-19,2025-01-06
9,CLM100009,PAT02344,Aetna,Physical Therapy,609.09,Paid,None,None,2024-04-29,2024-05-10


In [ ]:
from google.colab import files
files.download('claims_denial_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import auth
auth.authenticate_user()

project_id = 'steady-goods-509911-r7'  # copy this from the GCP console, not the display name

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=project_id)
client.create_dataset(f"{project_id}.claims_denial", exists_ok=True)

df = pd.read_csv('claims_denial_data.csv')
df.to_gbq(f"{project_id}.claims_denial.claims", project_id=project_id, if_exists='replace')
print("Loaded into BigQuery.")

/tmp/ipykernel_2937/1944379935.py:8: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq(f"{project_id}.claims_denial.claims", project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 7584.64it/s]

Loaded into BigQuery.
